- This notebook provides code for fine-tuning the medsam model, including the image encoder and prompt encoder, on an MRI Dataset.
- This is for experiments comparing end-to-end fine-tuning vs only fine-tuning the mask decoder portion of the network

In [ ]:
#%% setup environment
import numpy as np
import matplotlib.pyplot as plt
import os
join = os.path.join
from tqdm import tqdm
from skimage import transform
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.multiprocessing as mp
import monai

import torch.nn.functional as F
import argparse
import random
from datetime import datetime
import shutil
import glob
import pandas as pd
import nibabel as nib
import pickle
import time
import sys
sys.path.append('../../modified_medsam_repo')
from segment_anything import sam_model_registry
from MedSAM_HCP.dataset import MRIDataset, load_datasets, MRIDataset_Imgs_MedSAM
from MedSAM_HCP.MedSAM import MedSAM, logits_to_pred_probs
from MedSAM_HCP.build_sam import build_sam_vit_b_multiclass, resume_model_optimizer_and_epoch_from_checkpoint, save_model_optimizer_and_epoch_to_checkpoint
from MedSAM_HCP.utils_hcp import *
from MedSAM_HCP.loss_funcs_hcp import *
from MedSAM_HCP.logging_functions import init_wandb, print_cuda_memory, log_losses_step, log_predicted_probabilities, log_class_losses_as_barplots
from MedSAM_HCP.train_MedSAM_functions import retrieve_class_weights_tensor, train_step, validate_step, log_stuff_at_step, log_stuff_at_epoch


In [3]:
torch.manual_seed(2023)
torch.cuda.empty_cache()

In [4]:
pd.read_csv('/gpfs/data/luilab/karthik/pediatric_seg_proj/path_df_constant_bbox.csv')

,id,slice,image_embedding_slice_path,segmentation_slice_path,image_path,bbox_0,bbox_1,bbox_2,bbox_3
0,100206,0,/gpfs/data/cbi/hcp/hcp_ya/hcp_ya_slices_npy/pr...,/gpfs/data/cbi/hcp/hcp_ya/hcp_ya_slices_npy/se...,/gpfs/data/cbi/hcp/hcp_ya/hcp_ya_slices_npy/di...,0,0,256,256
1,100206,1,/gpfs/data/cbi/hcp/hcp_ya/hcp_ya_slices_npy/pr...,/gpfs/data/cbi/hcp/hcp_ya/hcp_ya_slices_npy/se...,/gpfs/data/cbi/hcp/hcp_ya/hcp_ya_slices_npy/di...,0,0,256,256
2,100206,2,/gpfs/data/cbi/hcp/hcp_ya/hcp_ya_slices_npy/pr...,/gpfs/data/cbi/hcp/hcp_ya/hcp_ya_slices_npy/se...,/gpfs/data/cbi/hcp/hcp_ya/hcp_ya_slices_npy/di...,0,0,256,256
3,100206,3,/gpfs/data/cbi/hcp/hcp_ya/hcp_ya_slices_npy/pr...,/gpfs/data/cbi/hcp/hcp_ya/hcp_ya_slices_npy/se...,/gpfs/data/cbi/hcp/hcp_ya/hcp_ya_slices_npy/di...,0,0,256,256
4,100206,4,/gpfs/data/cbi/hcp/hcp_ya/hcp_ya_slices_npy/pr...,/gpfs/data/cbi/hcp/hcp_ya/hcp_ya_slices_npy/se...,/gpfs/data/cbi/hcp/hcp_ya/hcp_ya_slices_npy/di...,0,0,256,256
...,...,...,...,...,...,...,...,...,...
284923,996782,251,/gpfs/data/cbi/hcp/hcp_ya/hcp_ya_slices_npy/pr...,/gpfs/data/cbi/hcp/hcp_ya/hcp_ya_slices_npy/se...,/gpfs/data/cbi/hcp/hcp_ya/hcp_ya_slices_npy/di...,0,0,256,256
284924,996782,252,/gpfs/data/cbi/hcp/hcp_ya/hcp_ya_slices_npy/pr...,/gpfs/data/cbi/hcp/hcp_ya/hcp_ya_slices_npy/se...,/gpfs/data/cbi/hcp/hcp_ya/hcp_ya_slices_npy/di...,0,0,256,256
284925,996782,253,/gpfs/data/cbi/hcp/hcp_ya/hcp_ya_slices_npy/pr...,/gpfs/data/cbi/hcp/hcp_ya/hcp_ya_slices_npy/se...,/gpfs/data/cbi/hcp/hcp_ya/hcp_ya_slices_npy/di...,0,0,256,256
284926,996782,254,/gpfs/data/cbi/hcp/hcp_ya/hcp_ya_slices_npy/pr...,/gpfs/data/cbi/hcp/hcp_ya/hcp_ya_slices_npy/se...,/gpfs/data/cbi/hcp/hcp_ya/hcp_ya_slices_npy/di...,0,0,256,256


In [110]:
class MRIDataset_Imgs_MedSAM(MRIDataset): 
    def __init__(self, data_frame, label_id=None, bbox_shift=0, label_converter=None, NUM_CLASSES = 256, as_one_hot = True, pool_labels = False, preprocess_fn=None):
        super().__init__(data_frame, label_id, bbox_shift, label_converter, NUM_CLASSES, as_one_hot, pool_labels, preprocess_fn)
    def __len__(self):
        return super().__len__()
    def __getitem__(self, index):
        # load image as npy (256x256x3)
        img_path = self.data_frame.loc[index,'image_path']
        img = Image.open(img_path)

        img_npy = np.array(img)

        if self.preprocess_fn is not None:
            img_npy = self.preprocess_fn(img_npy)
            
        img_npy = np.transpose(img_npy, axes = (2, 0, 1))

        img_slice_name = '_slice'.join(img_path.split('/')[-1:]).split('.png')[0]
        
        # load segmentation mask as npy
        seg_path = self.data_frame.loc[index,'segmentation_slice_path']
        if isinstance(seg_path, str) and os.path.exists(seg_path):
            seg_npy = np.load(seg_path) # (256, 256)
        else:
            seg_npy = np.full((256, 256), np.nan)
        
        if self.label_converter is not None:
            seg_npy = self.label_converter.hcp_to_compressed(seg_npy)
        
        if self.label_id is not None:
            if self.pool_labels:
                label_number = self.data_frame.loc[index,'label_number']
            else:
                label_number = self.label_id

            seg_npy = (seg_npy == label_number).astype(np.uint8)
            seg_tens = torch.tensor(seg_npy[None, :, :]).long()
        else:
            assert False

        # load bounding box coordinates from data frame
        x_min, x_max = self.data_frame.loc[index, 'bbox_0'], self.data_frame.loc[index, 'bbox_2']
        y_min, y_max = self.data_frame.loc[index, 'bbox_1'], self.data_frame.loc[index, 'bbox_3']
        
        if not np.any(np.isnan([x_min, x_max, y_min, y_max])): # if no nans
            # add perturbation to bounding box coordinates
            H, W = seg_npy.shape
            x_min = max(0, x_min - random.randint(0, self.bbox_shift))
            x_max = min(W, x_max + random.randint(0, self.bbox_shift))
            y_min = max(0, y_min - random.randint(0, self.bbox_shift))
            y_max = min(H, y_max + random.randint(0, self.bbox_shift))
        
        bboxes = np.array([x_min, y_min, x_max, y_max])

        return torch.tensor(img_npy).float(), seg_tens, torch.tensor(bboxes).float(), img_slice_name

    def get_slice_name(self, index):
        img_path = self.data_frame.loc[index,'image_path']
        ido = self.data_frame.loc[index,'id']
        sliceo = self.data_frame.loc[index,'slice']
        img_slice_name = f'{ido}_{sliceo}'
        return img_slice_name

    def load_image(self, index):
        return super().load_image(index, col_name='image_path')

In [143]:
def sam_img_to_embedding(x, model, device='cuda'):
    # x is an np array 3x256x256
    x = x.transpose(1, 2, 0)  # Change to 256x256x3
    x = transform.resize(x, (1024,1024), order=3, preserve_range=True, anti_aliasing=True).astype(np.uint8)
    x_tensor = torch.tensor(x).float()
    x_tensor_minmax = (x_tensor - x_tensor.min()) / np.clip(
                x_tensor.max() - x_tensor.min(), a_min=1e-8, a_max=None
            )
    x_tensor_minmax = x_tensor_minmax.float().permute(2,0,1).unsqueeze(0).to(device)
    embedding = model.image_encoder(x_tensor_minmax) # shape is (1, 256, 64, 64)

    # return as tensor
    return embedding

In [ ]:
NUM_CLASSES = 1
df_hcp = pd.read_csv('/gpfs/home/kn2347/HCP_MedSAM_project/modified_medsam_repo/hcp_mapping_processed.csv')
df_desired = pd.read_csv('/gpfs/home/kn2347/HCP_MedSAM_project/modified_medsam_repo/darts_name_class_mapping_processed.csv')
label_converter = LabelConverter(df_hcp, df_desired)
checkpoint='/gpfs/home/kn2347/HCP_MedSAM_project/modified_medsam_repo/medsam_vit_b.pth'
df_path ='/gpfs/data/luilab/karthik/pediatric_seg_proj/path_df_constant_bbox.csv'
train_test_split_path = '/gpfs/data/luilab/karthik/pediatric_seg_proj/train_val_test_split.pickle'
label_id = 1
df = pd.read_csv(df_path)
is_multitask = False
lr = 1e-4
image_encoder_lr=1e-4
prompt_encoder_lr=1e-4
weight_decay = 1e-2

loss_type = 'weighted_ce_dice_loss'
lambda_dice = 1
train_losses = []
val_losses = []
best_val_loss = 1e10
start_time = time.time()
total_number_of_training_examples_seen = 0
scaler = None
class_weights_tensor = torch.ones(1)


# build SAM model from checkpoint
sam_model = build_sam_vit_b_multiclass(num_classes=max(NUM_CLASSES, 3), checkpoint=checkpoint) # if single class, load original SAM model

# initialize MedSAM model object using the loaded SAM model
medsam_model = MedSAM(image_encoder=sam_model.image_encoder, 
                    mask_decoder=sam_model.mask_decoder,
                    prompt_encoder=sam_model.prompt_encoder,
                    multimask_output= is_multitask # 2 because unknown class is also present in single-task case
                ).cuda()

medsam_model.train()

# Setting up optimiser and loss func
params = list(
    medsam_model.parameters()
)

param_list = [
        {'params': medsam_model.image_encoder.parameters(), 'lr': image_encoder_lr, 'weight_decay': weight_decay},
        {'params': medsam_model.mask_decoder.parameters(), 'lr': lr, 'weight_decay': weight_decay},
        {'params': medsam_model.prompt_encoder.parameters(), 'lr': prompt_encoder_lr, 'weight_decay':weight_decay}
    ]
optimizer = torch.optim.AdamW(
    param_list
)
for param in params:
    param.requires_grad = True

In [136]:

print('Number of total parameters: ', sum(p.numel() for p in medsam_model.parameters())) 
print('Number of trainable parameters: ', sum(p.numel() for p in medsam_model.parameters() if p.requires_grad))
#params[0].requires_grad 

Number of total parameters:  93735472
Number of trainable parameters:  93735472


In [139]:
train_ds, val_ds, test_ds = load_datasets(df_path, train_test_split_path, label_id, bbox_shift=0, 
                sample_n_slices = None, label_converter=label_converter, NUM_CLASSES=NUM_CLASSES, 
                as_one_hot=True, pool_labels=False, preprocess_fn = None,
                dataset_type = MRIDataset_Imgs_MedSAM)

In [149]:
for epoch in range(0, 10):
    for step, (img, gt2D, boxes, _) in enumerate(tqdm(train_dataloader)):
        image_embedding = sam_img_to_embedding(img.numpy(), medsam_model)
        loss, class_losses, dice_class_losses, ce_class_losses, medsam_pred = train_step(
                medsam_model, optimizer, scaler, loss_type, class_weights_tensor, 
                lambda_dice, image_embedding, gt2D, boxes, args
            )

NameError: name 'train_dataloader' is not defined